https://docs.langchain.com/oss/python/integrations/document_loaders

In [2]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
loader =  WebBaseLoader(
    web_path=["https://www.educosys.com/course/genai"]
)

docs=loader.load()
print(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.educosys.com/course/genai', 'title': 'Hands-on Generative AI Course', 'description': 'Learn, Build, Deploy and Apply Generative AI', 'language': 'en'}, page_content="Hands-on Generative AI CourseCoursesBundle CoursesUpcoming CoursesMentorFree ContentTestimonialsFAQLogin Signup Ongoing LIVE CourseHands-on Generative AI CourseLearn, Build, Deploy and Apply Generative AIJoin Anytime!Get LifeTime AccessAccess all Live BatchesLifetime access of RecordingsAccess Discord CommunityCode availableBuild ProjectsLearn Future-Ready TechEnroll 1Week 1Foundations of Generative AI Introduction to AI Mathematical Foundations for AI Probability, Statistics, and Linear Algebra Basics of Neural Networks Gradient Descent and Optimization Basics Architectures: Feedforward, RNN, and CNN Mini Project - Build a Simple Neural Network Using TensorFlow Mini Project - Train an Autoencoder on the MNIST Dataset2Week 2Deep Generative Models Discriminative and Generative mode

https://docs.langchain.com/oss/python/langchain/rag

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

#print(all_splits)
print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 11 sub-documents.


In [12]:
from langchain_chroma import Chroma

In [14]:
from langchain_openai import OpenAIEmbeddings


embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(collection_name="educosys_genai_info", embedding_function=embeddings, persist_directory="./chroma_geni")
vectorstore.add_documents(documents=all_splits)
print(vectorstore._collection.count())  # Check total stored chunks


11


In [31]:

def retrieve_context(query: str):
  """Search for info related to educosys genai course"""
  try:
      embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
      for i, doc in enumerate(all_splits, start=1):
           vec = embeddings.embed_query(doc.page_content)  # embedding for this chunk
           print(f"\n--- Chunk {i} ---")
           print("Text snippet:", doc.page_content[:200], "...")
           print("Embedding length:", len(vec))
           print("Embedding preview (first 5 dims):", vec[:5])
      vector_store = Chroma(
          collection_name="educosys_genai_info",
          embedding_function=embeddings,
          persist_directory="./chroma_geni",
      )
      retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})


      print(f"Querying retrieve_context with: {query}")
      print("--------------------------------------------------------------")
      results = retriever.invoke(query)
      print(f"Retrieved documents: {len(results)} matches found")
      for i, doc in enumerate(results):
          print(f"Document {i + 1}: {doc.page_content[:100]}...")
    
      print("--------------------------------------------------------------")


      content = "\n".join([doc.page_content for doc in results])
      if not content:
          print(f"No content retrieved for query: {query}")
          return f"No reviews found for '{query}'."
    
      print("--------------------------------------------------------------")
      print(f"Returning content: {content[:200]}...")
      return content
  except Exception as e:
      print(f"Error in retrieve_context: {e}")
      return f"Error retrieving reviews for '{query}'. Please try again."
 

In [22]:
from langchain.chat_models import init_chat_model

In [23]:
llm = init_chat_model("gpt-4o", model_provider="openai")

In [25]:
from langgraph.prebuilt import create_react_agent

In [29]:
agent_executor = create_react_agent(llm, [retrieve_context])

/var/folders/tq/12nbb19j53q0mb2vftydl65m0000gn/T/ipykernel_45104/4122366744.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, [retrieve_context])


In [30]:

input_message = (
  "give me curriculcum of week 1 of educosys genai course?"
)
for event in agent_executor.stream(
  {"messages": [{"role": "user", "content": input_message}]},
  stream_mode="values"
):
  event["messages"][-1].pretty_print()


================================ Human Message =================================

give me curriculcum of week 1 of educosys genai course?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_QhQE72UVmTkKK1rW6Mmp1HtB)
 Call ID: call_QhQE72UVmTkKK1rW6Mmp1HtB
  Args:
    query: EducoSys GenAI course week 1 curriculum

--- Chunk 1 ---
Text snippet: Hands-on Generative AI CourseCoursesBundle CoursesUpcoming CoursesMentorFree ContentTestimonialsFAQLogin Signup Ongoing LIVE CourseHands-on Generative AI CourseLearn, Build, Deploy and Apply Generativ ...
Embedding length: 3072
Embedding preview (first 5 dims): [-0.02296033501625061, -0.008902516216039658, -0.012311171740293503, 0.0003981894697062671, -0.0010531822917982936]

--- Chunk 2 ---
Text snippet: Adversarial Networks (GANs) Variational Autoencoders (VAEs) Probabilistic Data Generation Using VAEs Four Mini Projects using TensorFlow Metrics Visualization using TensorBoard Mi